In [ ]:
import pandas as pd

DATA = '../data-public'
OUT = '../data'

TYPES = ['political', 'racial/ethnic', 'religious', 'gender/sexual', 'other']
MANIF = ['stereotype', 'vilification', 'dehumanization',
         'extreme_language', 'lack_of_empathy', 'invalidation']

parts = []
for split in ['train', 'dev', 'test']:
    d = pd.read_csv(f'{DATA}/{split}/eng.csv')
    d['split'] = split
    parts.append(d)

df = pd.concat(parts, ignore_index=True).drop(columns=['canary'], errors='ignore')
df.to_csv(f'{OUT}/polar_eng_all.csv', index=False)

print(df.shape)
print(df['split'].value_counts())

In [ ]:
overview = df.groupby('split').agg(n=('id', 'size'),
                                   polarized=('polarization', 'sum'))
overview.loc['total'] = overview.sum()
overview['polarized_%'] = (overview['polarized'] / overview['n'] * 100).round(1)
overview


In [ ]:
pol = df[df['polarization'] == 1]

labels = pd.DataFrame({
    'n_all':  df[TYPES + MANIF].sum(),
    'n_test': df[df['split'] == 'test'][TYPES + MANIF].sum(),
})
labels['%_of_polarized'] = (labels['n_all'] / len(pol) * 100).round(1)
labels.to_csv('../results/label_statistics.csv')
labels

In [ ]:
identity = ['racial/ethnic', 'religious', 'gender/sexual']
has_identity = pol[identity].sum(axis=1) > 0

print('Political, no identity type:    ', ((pol['political'] == 1) & ~has_identity).sum())
print('Political + identity type:      ', ((pol['political'] == 1) & has_identity).sum())
print('Identity type only (no political):', ((pol['political'] == 0) & has_identity).sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

short = ['stereo', 'vilif', 'dehum', 'extreme', 'no_empathy', 'invalid']
co = pol[MANIF].T.dot(pol[MANIF])
co.index = co.columns = short

plt.figure(figsize=(7, 5.5))
sns.heatmap(co, annot=True, fmt='d', cmap='Blues')
plt.title('Co-occurrence of manifestations (polarized texts)')
plt.tight_layout()
plt.savefig('../results/manifestation_cooccurrence.png', dpi=200)
plt.show()